# 04. 사용주기 AI 2단계 튜닝

1차 벤치마크에서 상위권이었던 모델을 대상으로 튜닝한 결과를 정리한다.

이번 단계의 선택 규칙은 다음과 같다.

1. `Train` split으로 후보 파라미터를 비교한다.
2. `Valid` split의 RMSE로 후보 파라미터를 선택한다.
3. 선택된 파라미터로 `Train + Valid`를 재학습한다.
4. `Test` split은 최종 평가에만 사용한다.

이 방식은 1차 벤치마크보다 실험 절차가 명확하고, 최종 산출물은 서버 호환 계약(`model_meta.json`, `model.pkl`)을 유지한다.

## 1. Import 및 경로

In [ ]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TERM_MONTHS = 6

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        data_path = path / "dataset" / "create_data" / "data_ml" / "phase4_training_data.csv"
        if data_path.exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "dataset" / "create_data" / "data_ml" / "phase4_training_data.csv"
EXPERIMENT_DIR = PROJECT_ROOT / "ai_model" / "experiments"
RUN_DIR = EXPERIMENT_DIR / "runs"
RESULT_PATH = EXPERIMENT_DIR / "stage2_valid_tuning_results.csv"

for directory in [EXPERIMENT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(DATA_PATH)

## 2. 데이터 로드 및 split

In [ ]:
FEATURES = [
    "내용연수",
    "취득금액",
    "부서가혹도",
    "가격민감도",
    "장비중요도",
    "리드타임등급",
    "취득월",
    "G2B목록명_Code",
    "물품분류명_Code",
    "운용부서코드_Code",
    "캠퍼스_Code",
]
TARGET = "실제수명_개월"

df = pd.read_csv(DATA_PATH)
df[TARGET] = df["실제수명"] * 12

train_mask = df["데이터세트구분"].eq("Train")
valid_mask = df["데이터세트구분"].eq("Valid")
train_valid_mask = df["데이터세트구분"].isin(["Train", "Valid"])
test_mask = df["데이터세트구분"].eq("Test")
pred_mask = df["데이터세트구분"].eq("Prediction")

X_train = df.loc[train_mask, FEATURES]
y_train = df.loc[train_mask, TARGET]
X_valid = df.loc[valid_mask, FEATURES]
y_valid = df.loc[valid_mask, TARGET]
X_train_valid = df.loc[train_valid_mask, FEATURES]
y_train_valid = df.loc[train_valid_mask, TARGET]
X_test = df.loc[test_mask, FEATURES]
y_test = df.loc[test_mask, TARGET]

test_age_months = df.loc[test_mask, "운용연차"].to_numpy() * 12
actual_rul_test = y_test.to_numpy() - test_age_months
actual_term_fail_test = actual_rul_test <= TERM_MONTHS

display(df["데이터세트구분"].value_counts().to_frame("count"))

## 3. Valid 탐색 결과 반영 파라미터

아래 파라미터는 후보 탐색에서 `Valid RMSE`가 가장 좋았던 값이다. 재현 시간을 줄이기 위해 탐색 전체가 아니라 선택된 파라미터부터 재학습한다.

In [ ]:
BEST_CONFIGS = {
    "CatBoost": {
        "valid_rmse_months": 9.272885207090896,
        "params": {
            "iterations": 800,
            "depth": 7,
            "learning_rate": 0.05,
            "l2_leaf_reg": 5,
            "subsample": 1.0,
        },
    },
    "XGBoost": {
        "valid_rmse_months": 9.229518646938768,
        "params": {
            "n_estimators": 500,
            "max_depth": 3,
            "learning_rate": 0.1,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "min_child_weight": 1,
            "reg_lambda": 1.0,
        },
    },
    "RandomForest": {
        "valid_rmse_months": 9.953720044817448,
        "params": {
            "n_estimators": 200,
            "max_depth": 10,
            "min_samples_split": 5,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
        },
    },
    "ExtraTrees": {
        "valid_rmse_months": 9.801557513938672,
        "params": {
            "n_estimators": 500,
            "max_depth": 10,
            "min_samples_split": 10,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
            "bootstrap": False,
        },
    },
}

## 4. 학습, 평가, 저장 함수

In [ ]:
def make_model(model_name: str, params: dict):
    if model_name == "CatBoost":
        estimator = CatBoostRegressor(
            random_seed=RANDOM_STATE,
            loss_function="RMSE",
            verbose=False,
            allow_writing_files=False,
            **params,
        )
    elif model_name == "XGBoost":
        estimator = XGBRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1,
            objective="reg:squarederror",
            **params,
        )
    elif model_name == "RandomForest":
        estimator = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    elif model_name == "ExtraTrees":
        estimator = ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    return make_pipeline(SimpleImputer(strategy="median"), estimator)

def evaluate_test(model_name: str, y_pred, valid_rmse: float, params: dict) -> dict:
    pred_rul = y_pred - test_age_months
    pred_term_fail = pred_rul <= TERM_MONTHS

    return {
        "model": model_name,
        "valid_rmse_months": float(valid_rmse),
        "test_rmse_months": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "test_mae_months": float(mean_absolute_error(y_test, y_pred)),
        "test_r2": float(r2_score(y_test, y_pred)),
        "term_precision": float(precision_score(actual_term_fail_test, pred_term_fail, zero_division=0)),
        "term_recall": float(recall_score(actual_term_fail_test, pred_term_fail, zero_division=0)),
        "term_f1": float(f1_score(actual_term_fail_test, pred_term_fail, zero_division=0)),
        "best_params_json": json.dumps(params, ensure_ascii=False),
    }

def extract_feature_importance(model):
    estimator = model.steps[-1][1]
    if not hasattr(estimator, "feature_importances_"):
        return None
    return (
        pd.DataFrame({"feature": FEATURES, "importance": estimator.feature_importances_})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

## 5. Train+Valid 재학습 및 Test 평가

In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
STAGE2_DIR = RUN_DIR / f"{RUN_ID}_stage2_valid_tuning"
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

rows = []
trained_models = {}

for model_name, config in BEST_CONFIGS.items():
    print(f"[TRAIN] {model_name}")
    params = config["params"]
    model = make_model(model_name, params)
    model.fit(X_train_valid, y_train_valid)
    y_pred = model.predict(X_test)

    metrics = evaluate_test(model_name, y_pred, config["valid_rmse_months"], params)
    metrics["run_id"] = RUN_ID
    rows.append(metrics)
    trained_models[model_name] = model

    model_dir = STAGE2_DIR / model_name.lower()
    model_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, model_dir / "model.pkl")

    model_meta = {
        "run_id": RUN_ID,
        "stage": "stage2_valid_tuning",
        "model_name": model_name,
        "target": TARGET,
        "features": FEATURES,
        "rmse_months": metrics["test_rmse_months"],
        "mae_months": metrics["test_mae_months"],
        "r2": metrics["test_r2"],
        "valid_rmse_months": metrics["valid_rmse_months"],
        "best_params": params,
        "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "split_rule": "Train parameter selection, Train+Valid retrain, Test final evaluation",
    }
    with open(model_dir / "model_meta.json", "w", encoding="utf-8") as f:
        json.dump(model_meta, f, ensure_ascii=False, indent=2)
    with open(model_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    pred_df = df.loc[test_mask, ["물품고유번호", "G2B목록명", "운용연차", "실제수명"]].copy()
    pred_df["actual_total_life_months"] = y_test.to_numpy()
    pred_df["pred_total_life_months"] = y_pred
    pred_df["actual_rul_months"] = actual_rul_test
    pred_df["pred_rul_months"] = y_pred - test_age_months
    pred_df.to_csv(model_dir / "test_predictions.csv", index=False, encoding="utf-8-sig")

    fi_df = extract_feature_importance(model)
    if fi_df is not None:
        fi_df.to_csv(model_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")
        plt.figure(figsize=(9, 5))
        sns.barplot(data=fi_df, x="importance", y="feature", palette="mako")
        plt.title(f"Feature Importance - {model_name}")
        plt.tight_layout()
        plt.savefig(model_dir / "feature_importance.png", dpi=150, bbox_inches="tight")
        plt.show()

results_df = pd.DataFrame(rows).sort_values("test_rmse_months").reset_index(drop=True)
results_df["artifact_dir"] = results_df["model"].apply(lambda name: str((STAGE2_DIR / name.lower()).relative_to(PROJECT_ROOT)))
results_df.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
results_df.to_csv(STAGE2_DIR / "stage2_valid_tuning_results.csv", index=False, encoding="utf-8-sig")

display(results_df)
print(f"Stage 2 dir: {STAGE2_DIR}")

## 6. Prediction split 스모크 테스트

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
df_predict = df.loc[pred_mask].copy()

pred_total_months = best_model.predict(df_predict[FEATURES])
raw_rul = pred_total_months - df_predict["운용연차"].to_numpy() * 12
rng = np.random.default_rng(RANDOM_STATE)
zombie_months = rng.uniform(1.0, 4.0, size=len(raw_rul))
rul_months = np.round(np.where(raw_rul <= 0.5, zombie_months, raw_rul), 1)

smoke_summary = {
    "best_model": best_model_name,
    "prediction_rows": int(len(df_predict)),
    "term_failure_count": int((rul_months <= TERM_MONTHS).sum()),
    "min_rul_months": float(np.min(rul_months)),
    "max_rul_months": float(np.max(rul_months)),
}

with open(STAGE2_DIR / "best_smoke_summary.json", "w", encoding="utf-8") as f:
    json.dump(smoke_summary, f, ensure_ascii=False, indent=2)

display(pd.DataFrame([smoke_summary]))

## 7. 배포 경로 교체 셀

아래 셀은 기본값이 `False`라서 기존 서버 호환 모델을 덮어쓰지 않는다. 최종 확정 후에만 `True`로 바꾼다.

In [ ]:
DEPLOY_TO_SERVER_COMPAT_PATH = False

if DEPLOY_TO_SERVER_COMPAT_PATH:
    deploy_dir = PROJECT_ROOT / "ai_model" / "saved_models" / "random_forest"
    deploy_dir.mkdir(parents=True, exist_ok=True)
    best_model_dir = STAGE2_DIR / best_model_name.lower()
    joblib.dump(best_model, deploy_dir / "rf_final_model.pkl")
    with open(best_model_dir / "model_meta.json", "r", encoding="utf-8") as f:
        best_meta = json.load(f)
    with open(deploy_dir / "model_meta.json", "w", encoding="utf-8") as f:
        json.dump(best_meta, f, ensure_ascii=False, indent=2)
    print(f"배포 호환 경로 저장 완료: {deploy_dir}")
else:
    print("배포 경로는 변경하지 않았습니다.")